# J10g — Tourisme et hébergements en Île-de-France

## Objectifs
- Extraire les observations touristiques de la BPE 2025.
- Distinguer hébergements et services touristiques.
- Calculer nombres, parts des groupes et parts intragroupe.
- Calculer les densités uniquement par groupe.
- Documenter les capacités d'hébergement et leur complétude.
- Enrichir le profil communal J10f.

## Groupes d'analyse
Les groupes "Hébergements touristiques" et "Services touristiques"
sont construits pour le projet. Ils ne sont pas des sous-domaines
officiels supplémentaires de la BPE.

## Limites
- Comptages d'observations BPE.
- Ouverture actuelle non vérifiée.
- Aucune mesure de fréquentation, de nuitées ou de taux d'occupation.
- Chambres, emplacements et logements conservés séparément.
- Le périmètre ne couvre pas toutes les formes de location touristique.

In [1]:
#Import des bibliothèques

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)

Pandas : 2.2.2


In [2]:
#Définir les chemins

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_INTERIM = RACINE / "data" / "interim" / "j10g"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

FICHIER_BPE = (
    RACINE / "data" / "raw" / "bpe" / "BPE25.csv"
)

FICHIER_PROFIL = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10f.csv"
)

FICHIER_METADATA_J10F = (
    RACINE / "data" / "interim" / "j10f" / "metadata_j10f.json"
)

FICHIER_DETAIL = (
    DOSSIER_PROCESSED / "tourisme_hebergements_idf_j10g.csv"
)

FICHIER_COMMUNES = (
    DOSSIER_INTERIM / "tourisme_hebergements_communes.csv"
)

FICHIER_SORTIE = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10g.csv"
)

for fichier in [FICHIER_BPE, FICHIER_PROFIL]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Profil d'entrée :", FICHIER_PROFIL)
print("Profil de sortie :", FICHIER_SORTIE)

Profil d'entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10f.csv
Profil de sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10g.csv


In [3]:
#Fonctions utiles

def normaliser_nom_colonne(nom):
    nom = unicodedata.normalize(
        "NFKD", str(nom).strip().upper()
    )
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^A-Z0-9]+", "_", nom).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    manquants = {
        "", "NA", "NAN", "NONE", "NULL",
        "<NA>", "_U", "_Z", "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(manquants)
    )


def normaliser_code_commune(serie):
    resultat = nettoyer_texte(serie).str.upper()
    resultat = resultat.str.replace(r"\.0$", "", regex=True)

    masque = resultat.str.fullmatch(r"\d{1,5}", na=False)

    return resultat.where(
        ~masque,
        resultat.str.zfill(5),
    )


def convertir_nombre(serie):
    return pd.to_numeric(
        nettoyer_texte(serie)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def sauvegarder_csv(table, chemin):
    table.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig",
    )


def sha256_fichier(chemin):
    empreinte = hashlib.sha256()

    with open(chemin, "rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    return empreinte.hexdigest()

In [4]:
#Charger et controler le profil J10f

profil = pd.read_csv(
    FICHIER_PROFIL,
    dtype="string",
    encoding="utf-8-sig",
)

profil.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil.columns
]

if profil.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans le profil.")

if "CODGEO" not in profil.columns:
    raise ValueError("La colonne CODGEO est absente.")

profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

if not profil["CODGEO"].str.fullmatch(r"\d{5}", na=False).all():
    raise ValueError("Codes communaux manquants ou invalides.")

if not profil["CODGEO"].is_unique:
    raise ValueError("Plusieurs lignes existent pour une commune.")

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91", "92", "93", "94", "95"
}

if not profil["CODGEO"].str[:2].isin(DEPARTEMENTS_IDF).all():
    raise ValueError("Codes hors Île-de-France dans le profil.")

if "NB_OBS_EDUCATION_BPE2025" not in profil.columns:
    raise ValueError("Le total attendu du J10f est absent.")

codes_profil = set(profil["CODGEO"])

print("Communes :", len(profil))
print("Colonnes initiales :", len(profil.columns))

display(profil.head())

Communes : 1266
Colonnes initiales : 945


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,NB_OBS_RESIDENCES_UNIVERSITAIRES_BPE2025,NB_OBS_RESTAURANTS_UNIVERSITAIRES_BPE2025,NB_OBS_ENSEIGNEMENT_PREMIER_DEGRE_BPE2025,NB_OBS_ENSEIGNEMENT_COLLEGES_BPE2025,NB_OBS_ENSEIGNEMENT_LYCEES_SECTIONS_BPE2025,NB_OBS_SUPERIEUR_NON_UNIVERSITAIRE_BPE2025,NB_OBS_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_BPE2025,NB_OBS_FORMATION_CONTINUE_APPRENTISSAGE_BPE2025,NB_OBS_SERVICES_UNIVERSITAIRES_BPE2025,NB_OBS_EDUCATION_BPE2025,PART_ENSEIGNEMENT_PREMIER_DEGRE_J10F_PCT,PART_ENSEIGNEMENT_COLLEGES_J10F_PCT,PART_ENSEIGNEMENT_LYCEES_SECTIONS_J10F_PCT,PART_SUPERIEUR_NON_UNIVERSITAIRE_J10F_PCT,PART_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_J10F_PCT,PART_FORMATION_CONTINUE_APPRENTISSAGE_J10F_PCT,PART_SERVICES_UNIVERSITAIRES_J10F_PCT,PART_ECOLES_MATERNELLES_DANS_GROUPE_PCT,PART_ECOLES_MIXTES_MATERNELLE_ELEMENTAIRE_DANS_GROUPE_PCT,PART_ECOLES_ELEMENTAIRES_DANS_GROUPE_PCT,PART_COLLEGES_DANS_GROUPE_PCT,PART_LYCEES_GENERAUX_TECHNOLOGIQUES_DANS_GROUPE_PCT,PART_LYCEES_PROFESSIONNELS_DANS_GROUPE_PCT,PART_LYCEES_ET_ETABLISSEMENTS_SECONDAIRES_AGRICOLES_DANS_GROUPE_PCT,PART_SECTIONS_GENERALES_TECHNOLOGIQUES_DANS_GROUPE_PCT,PART_SECTIONS_PROFESSIONNELLES_DANS_GROUPE_PCT,PART_FORMATIONS_STS_CPGE_DANS_GROUPE_PCT,PART_FORMATIONS_COMMERCE_GESTION_DANS_GROUPE_PCT,PART_AUTRES_FORMATIONS_POSTBAC_NON_UNIVERSITAIRES_DANS_GROUPE_PCT,PART_ECOLES_FORMATIONS_SANITAIRES_SOCIALES_DANS_GROUPE_PCT,PART_ECOLES_METIERS_CULTURE_DANS_GROUPE_PCT,PART_UFR_COMPOSANTES_DANS_GROUPE_PCT,PART_INSTITUTS_UNIVERSITAIRES_DANS_GROUPE_PCT,PART_ECOLES_INGENIEURS_DANS_GROUPE_PCT,PART_ENSEIGNEMENT_SUPERIEUR_GENERAL_PRIVE_DANS_GROUPE_PCT,PART_ENSEIGNEMENT_SUPERIEUR_AGRICOLE_DANS_GROUPE_PCT,PART_AUTRES_STRUCTURES_ENSEIGNEMENT_SUPERIEUR_DANS_GROUPE_PCT,PART_GRETA_DANS_GROUPE_PCT,PART_FORMATION_CONTINUE_AGRICOLE_DANS_GROUPE_PCT,PART_FORMATIONS_METIERS_SPORT_DANS_GROUPE_PCT,PART_ORGANISMES_FORMATION_APPRENTISSAGE_DANS_GROUPE_PCT,PART_RESIDENCES_UNIVERSITAIRES_DANS_GROUPE_PCT,PART_RESTAURANTS_UNIVERSITAIRES_DANS_GROUPE_PCT,DENSITE_OBS_ENSEIGNEMENT_PREMIER_DEGRE_10000_HAB,DENSITE_OBS_ENSEIGNEMENT_COLLEGES_10000_HAB,DENSITE_OBS_ENSEIGNEMENT_LYCEES_SECTIONS_10000_HAB,DENSITE_OBS_SUPERIEUR_NON_UNIVERSITAIRE_10000_HAB,DENSITE_OBS_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_10000_HAB,DENSITE_OBS_FORMATION_CONTINUE_APPRENTISSAGE_10000_HAB,DENSITE_OBS_SERVICES_UNIVERSITAIRES_10000_HAB
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.414592524,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.93055085738077,39.40474191053151,59.38860910108081,39.8307111089857,60.03060590120033,200.61215141394868,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,91,59,841,218,241,466,312,666,150,2894,29.060124395300623,7.532

In [5]:
#Définir les catégories et groupes

TYPES = {
    "G101": "AGENCES_VOYAGES",
    "G102": "HOTELS",
    "G103": "CAMPINGS",
    "G104": "INFORMATION_TOURISTIQUE",
    "G105": "AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES",
}

GROUPES = {
    "HEBERGEMENTS_TOURISTIQUES": ["G102", "G103", "G105"],
    "SERVICES_TOURISTIQUES": ["G101", "G104"],
}

TYPE_VERS_GROUPE = {
    code: groupe
    for groupe, codes in GROUPES.items()
    for code in codes
}

codes_groupes = [
    code
    for codes in GROUPES.values()
    for code in codes
]

assert len(codes_groupes) == len(set(codes_groupes))
assert set(codes_groupes) == set(TYPES)

referentiel = pd.DataFrame({
    "TYPEQU": list(TYPES),
    "CATEGORIE_ANALYSE": list(TYPES.values()),
})

referentiel["GROUPE_ANALYSE"] = (
    referentiel["TYPEQU"].map(TYPE_VERS_GROUPE)
)

referentiel["ORIGINE_GROUPE"] = "REGROUPEMENT_ANALYTIQUE_PROJET"

sauvegarder_csv(
    referentiel,
    DOSSIER_INTERIM / "referentiel_tourisme.csv",
)

display(referentiel)

,TYPEQU,CATEGORIE_ANALYSE,GROUPE_ANALYSE,ORIGINE_GROUPE
0,G101,AGENCES_VOYAGES,SERVICES_TOURISTIQUES,REGROUPEMENT_ANALYTIQUE_PROJET
1,G102,HOTELS,HEBERGEMENTS_TOURISTIQUES,REGROUPEMENT_ANALYTIQUE_PROJET
2,G103,CAMPINGS,HEBERGEMENTS_TOURISTIQUES,REGROUPEMENT_ANALYTIQUE_PROJET
3,G104,INFORMATION_TOURISTIQUE,SERVICES_TOURISTIQUES,REGROUPEMENT_ANALYTIQUE_PROJET
4,G105,AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES,HEBERGEMENTS_TOURISTIQUES,REGROUPEMENT_ANALYTIQUE_PROJET


In [6]:
#Vérifier le format du BPE

with open(
    FICHIER_BPE,
    "r",
    encoding="utf-8-sig",
) as flux:
    premiere_ligne = flux.readline()

separateurs_valides = []

for separateur in [";", ",", "\t", "|"]:
    colonnes = next(
        csv.reader([premiere_ligne], delimiter=separateur)
    )

    colonnes = [
        normaliser_nom_colonne(colonne)
        for colonne in colonnes
    ]

    if {"AN", "DEPCOM", "TYPEQU"}.issubset(colonnes):
        separateurs_valides.append(separateur)

if len(separateurs_valides) != 1:
    raise ValueError(
        "En-tête BPE détaillé non reconnu. "
        "AN, DEPCOM et TYPEQU sont attendus."
    )

SEPARATEUR_BPE = separateurs_valides[0]

entete = pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding="utf-8-sig",
    nrows=0,
)

noms = [
    normaliser_nom_colonne(colonne)
    for colonne in entete.columns
]

if len(noms) != len(set(noms)):
    raise ValueError("Colonnes ambiguës après normalisation.")

print("Séparateur :", repr(SEPARATEUR_BPE))

Séparateur : ';'


In [7]:
#Extraire les informations par blocs

blocs_retenus = []
nombre_lignes_lues = 0
codes_inconnus = set()

with pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    dtype="string",
    encoding="utf-8-sig",
    keep_default_na=False,
    chunksize=20_000,
) as lecteur:

    for numero_bloc, bloc in enumerate(lecteur, start=1):
        bloc.columns = [
            normaliser_nom_colonne(c) for c in bloc.columns
        ]

        bloc["RANG_SOURCE_J10G"] = np.arange(
            nombre_lignes_lues + 1,
            nombre_lignes_lues + len(bloc) + 1,
        )

        nombre_lignes_lues += len(bloc)

        types = nettoyer_texte(bloc["TYPEQU"]).str.upper()
        communes = normaliser_code_commune(bloc["DEPCOM"])

        est_idf = communes.str[:2].isin(DEPARTEMENTS_IDF)
        est_tourisme = types.str.startswith("G", na=False)

        inconnus = est_idf & est_tourisme & ~types.isin(TYPES)

        codes_inconnus.update(
            types.loc[inconnus].dropna().tolist()
        )

        masque = est_idf & types.isin(TYPES)
        selection = bloc.loc[masque].copy()

        if not selection.empty:
            selection["TYPEQU_ANALYSE"] = types.loc[masque]
            selection["CODGEO_SOURCE"] = communes.loc[masque]
            blocs_retenus.append(selection)

        if numero_bloc % 25 == 0:
            print(f"{nombre_lignes_lues:,} lignes examinées")

if codes_inconnus:
    raise ValueError(
        "Codes touristiques à vérifier : "
        + ", ".join(sorted(codes_inconnus))
    )

if not blocs_retenus:
    raise ValueError("Aucune observation touristique retenue.")

tourisme = pd.concat(blocs_retenus, ignore_index=True)
del blocs_retenus

if not nettoyer_texte(tourisme["AN"]).eq("2025").fillna(False).all():
    raise ValueError("Millésime différent de 2025 dans l'extraction.")

print("Observations retenues :", len(tourisme))

500,000 lignes examinées
1,000,000 lignes examinées
1,500,000 lignes examinées
2,000,000 lignes examinées
2,500,000 lignes examinées
Observations retenues : 9120


In [8]:
#Ajouter la tracabilité

HASH_BPE = sha256_fichier(FICHIER_BPE)
DATE_TRAITEMENT = datetime.now(timezone.utc).isoformat()

tourisme["ID_OBSERVATION"] = (
    "BPE2025_"
    + HASH_BPE[:16]
    + "_"
    + tourisme["RANG_SOURCE_J10G"].astype("string")
)

tourisme["CATEGORIE_ANALYSE"] = (
    tourisme["TYPEQU_ANALYSE"].map(TYPES)
)

tourisme["GROUPE_ANALYSE"] = (
    tourisme["TYPEQU_ANALYSE"].map(TYPE_VERS_GROUPE)
)

tourisme["SOURCE_DONNEES"] = "INSEE_BPE_2025"
tourisme["STATUT_OUVERTURE_ACTUELLE"] = "NON_VERIFIE"

assert tourisme["ID_OBSERVATION"].is_unique
assert tourisme["GROUPE_ANALYSE"].notna().all()

display(tourisme["CATEGORIE_ANALYSE"].value_counts())

CATEGORIE_ANALYSE
INFORMATION_TOURISTIQUE                        3571
HOTELS                                         2699
AGENCES_VOYAGES                                2498
AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES     272
CAMPINGS                                         80
Name: count, dtype: int64

In [9]:
#Harmoniser les communes

tourisme["CODGEO"] = tourisme["CODGEO_SOURCE"]

arrondissements_paris = {
    f"751{numero:02d}" for numero in range(1, 21)
}

if codes_profil & arrondissements_paris:
    raise ValueError(
        "Le profil doit être communal, sans arrondissements parisiens."
    )

if "75056" in codes_profil:
    tourisme.loc[
        tourisme["CODGEO"].isin(arrondissements_paris),
        "CODGEO",
    ] = "75056"

if "93066" in codes_profil and "93059" not in codes_profil:
    tourisme.loc[
        tourisme["CODGEO"].eq("93059"),
        "CODGEO",
    ] = "93066"

hors_profil = tourisme.loc[
    ~tourisme["CODGEO"].isin(codes_profil)
].copy()

sauvegarder_csv(
    hors_profil,
    DOSSIER_INTERIM / "tourisme_hors_profil.csv",
)

print("Observations hors profil :", len(hors_profil))

if not hors_profil.empty:
    display(
        hors_profil[
            ["CODGEO_SOURCE", "CODGEO", "TYPEQU_ANALYSE"]
        ].drop_duplicates()
    )

    raise ValueError(
        "Résoudre les codes non appariés avant de poursuivre."
    )

Observations hors profil : 0


In [10]:
#Préparer les noms et adresses

colonnes_facultatives = [
    "SIRET", "NOMRS", "CNOMRS",
    "NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR",
    "LATITUDE", "LONGITUDE",
    "CAPACITE", "INDIC_CAPA",
]

for colonne in colonnes_facultatives:
    if colonne not in tourisme.columns:
        tourisme[colonne] = pd.Series(
            pd.NA,
            index=tourisme.index,
            dtype="string",
        )

tourisme["NOM_AFFICHAGE"] = (
    nettoyer_texte(tourisme["NOMRS"])
    .fillna(nettoyer_texte(tourisme["CNOMRS"]))
)

parties_adresse = tourisme[
    ["NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR"]
].apply(nettoyer_texte)

tourisme["ADRESSE_RECOMPOSEE"] = (
    parties_adresse.fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

tourisme["SIRET_NETTOYE"] = nettoyer_texte(tourisme["SIRET"])

siret_conforme = (
    tourisme["SIRET_NETTOYE"].str.fullmatch(r"\d{14}", na=False)
    & tourisme["SIRET_NETTOYE"].ne("0" * 14).fillna(False)
)

tourisme["SIRET_TYPE_REPETE"] = (
    siret_conforme
    & tourisme.duplicated(
        subset=["SIRET_NETTOYE", "TYPEQU_ANALYSE"],
        keep=False,
    )
)

repetitions = tourisme.loc[
    tourisme["SIRET_TYPE_REPETE"]
].copy()

sauvegarder_csv(
    repetitions,
    DOSSIER_INTERIM / "siret_types_repetes_a_examiner.csv",
)

print("Observations avec SIRET et type répétés :", len(repetitions))
print("Aucune suppression automatique.")

Observations avec SIRET et type répétés : 23
Aucune suppression automatique.


In [11]:
#Controler les coordonnées

tourisme["LATITUDE_NUM"] = convertir_nombre(
    tourisme["LATITUDE"]
)

tourisme["LONGITUDE_NUM"] = convertir_nombre(
    tourisme["LONGITUDE"]
)

coordonnees_presentes = (
    tourisme["LATITUDE_NUM"].notna()
    & tourisme["LONGITUDE_NUM"].notna()
)

coordonnees_plausibles = (
    tourisme["LATITUDE_NUM"].between(48.0, 49.3)
    & tourisme["LONGITUDE_NUM"].between(1.4, 3.7)
).fillna(False)

tourisme["STATUT_COORDONNEES"] = "ABSENTES_OU_NON_NUMERIQUES"

tourisme.loc[
    coordonnees_presentes,
    "STATUT_COORDONNEES",
] = "HORS_RECTANGLE_CONTROLE"

tourisme.loc[
    coordonnees_plausibles,
    "STATUT_COORDONNEES",
] = "PLAUSIBLES_A_VERIFIER"

controle_coordonnees = (
    tourisme.groupby(
        ["CATEGORIE_ANALYSE", "STATUT_COORDONNEES"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    controle_coordonnees,
    DOSSIER_INTERIM / "controle_coordonnees.csv",
)

display(controle_coordonnees)

,CATEGORIE_ANALYSE,STATUT_COORDONNEES,NB_OBSERVATIONS
0,AGENCES_VOYAGES,ABSENTES_OU_NON_NUMERIQUES,156
1,AGENCES_VOYAGES,PLAUSIBLES_A_VERIFIER,2342
2,AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES,PLAUSIBLES_A_VERIFIER,272
3,CAMPINGS,PLAUSIBLES_A_VERIFIER,80
4,HOTELS,ABSENTES_OU_NON_NUMERIQUES,5
5,HOTELS,PLAUSIBLES_A_VERIFIER,2694
6,INFORMATION_TOURISTIQUE,ABSENTES_OU_NON_NUMERIQUES,882
7,INFORMATION_TOURISTIQUE,PLAUSIBLES_A_VERIFIER,2689


In [12]:
#Préparer les capacités avec leurs unités

UNITES_CAPACITE = {
    "G102": "CHAMBRES",
    "G103": "EMPLACEMENTS_PASSAGE",
    "G105": "LOGEMENTS",
}

capacite = convertir_nombre(tourisme["CAPACITE"])

capacite_valide = (
    capacite.notna()
    & capacite.ge(0)
    & capacite.mod(1).eq(0)
).fillna(False)

concerne_capacite = nettoyer_texte(
    tourisme["INDIC_CAPA"]
).eq("1").fillna(False)

est_hebergement = tourisme["TYPEQU_ANALYSE"].isin(
    UNITES_CAPACITE
)

tourisme["UNITE_CAPACITE_ANALYSE"] = (
    tourisme["TYPEQU_ANALYSE"].map(UNITES_CAPACITE)
)

tourisme["CAPACITE_NUM"] = capacite.where(
    est_hebergement & concerne_capacite & capacite_valide
)

tourisme["STATUT_CAPACITE_ANALYSE"] = "HORS_CHAMP"

tourisme.loc[
    est_hebergement,
    "STATUT_CAPACITE_ANALYSE",
] = "MANQUANTE_OU_NON_EXPLOITABLE"

tourisme.loc[
    tourisme["CAPACITE_NUM"].notna(),
    "STATUT_CAPACITE_ANALYSE",
] = "RENSEIGNEE"

display(
    tourisme[
        [
            "CATEGORIE_ANALYSE",
            "NOM_AFFICHAGE",
            "CAPACITE_NUM",
            "UNITE_CAPACITE_ANALYSE",
            "STATUT_CAPACITE_ANALYSE",
        ]
    ].head(15)
)

,CATEGORIE_ANALYSE,NOM_AFFICHAGE,CAPACITE_NUM,UNITE_CAPACITE_ANALYSE,STATUT_CAPACITE_ANALYSE
0,AGENCES_VOYAGES,MAKILA,<NA>,NaN,HORS_CHAMP
1,AGENCES_VOYAGES,VACANCES HELIADES,<NA>,NaN,HORS_CHAMP
2,AGENCES_VOYAGES,SOCIETE DE PROMOTION TOURISTIQUE ET ECONOMIQUE,<NA>,NaN,HORS_CHAMP
3,AGENCES_VOYAGES,NIPPON TRAVEL AGENCY FRANCE,<NA>,NaN,HORS_CHAMP
4,AGENCES_VOYAGES,JALPAK INTERNATIONAL (FRANCE) SAS,<NA>,NaN,HORS_CHAMP
5,AGENCES_VOYAGES,SABERATOURS,<NA>,NaN,HORS_CHAMP
6,AGENCES_VOYAGES,NOTRE DAME DE LA PAIX,<NA>,NaN,HORS_CHAMP
7,AGENCES_VOYAGES,AUTRE VOYAGE,<NA>,NaN,HORS_CHAMP
8,AGENCES_VOYAGES,EXCLUSIF COMMUNICATION,<NA>,NaN,HORS_CHAMP
9,AGENCES_VOYAGES,SOC SERVICES TOURISTIQUES,<NA>,NaN,HORS_CHAMP


In [13]:
#Compter par catégorie et groupe

COLONNES_TYPES = {
    code: f"NB_OBS_{libelle}_BPE2025"
    for code, libelle in TYPES.items()
}

COLONNES_GROUPES = {
    groupe: f"NB_OBS_{groupe}_BPE2025"
    for groupe in GROUPES
}

comptages = pd.crosstab(
    tourisme["CODGEO"],
    tourisme["TYPEQU_ANALYSE"],
)

comptages = (
    comptages.reindex(columns=list(TYPES), fill_value=0)
    .rename(columns=COLONNES_TYPES)
    .reset_index()
)

indicateurs = profil[["CODGEO"]].merge(
    comptages,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonnes_types = list(COLONNES_TYPES.values())

indicateurs[colonnes_types] = (
    indicateurs[colonnes_types]
    .fillna(0)
    .astype("int64")
)

for groupe, codes in GROUPES.items():
    colonnes_a_sommer = [
        COLONNES_TYPES[code] for code in codes
    ]

    indicateurs[COLONNES_GROUPES[groupe]] = (
        indicateurs[colonnes_a_sommer].sum(axis=1)
    )

COLONNE_TOTAL = "NB_OBS_TOURISME_BPE2025"

indicateurs[COLONNE_TOTAL] = (
    indicateurs[colonnes_types].sum(axis=1)
)

assert indicateurs[COLONNE_TOTAL].sum() == len(tourisme)

assert (
    indicateurs[list(COLONNES_GROUPES.values())].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

print("Comptages contrôlés.")

Comptages contrôlés.


In [14]:
#Calculer les parts

total = indicateurs[COLONNE_TOTAL].replace(0, np.nan)

PARTS_GROUPES = {}
PARTS_INTRAGROUPE = {}
series_parts = {}

for groupe in GROUPES:
    nom_part = f"PART_{groupe}_J10G_PCT"

    series_parts[nom_part] = (
        indicateurs[COLONNES_GROUPES[groupe]]
        .div(total)
        .mul(100)
    )

    PARTS_GROUPES[groupe] = nom_part

for code, libelle in TYPES.items():
    groupe = TYPE_VERS_GROUPE[code]
    nom_part = f"PART_{libelle}_DANS_GROUPE_PCT"

    total_groupe = indicateurs[
        COLONNES_GROUPES[groupe]
    ].replace(0, np.nan)

    series_parts[nom_part] = (
        indicateurs[COLONNES_TYPES[code]]
        .div(total_groupe)
        .mul(100)
    )

    PARTS_INTRAGROUPE[code] = nom_part

indicateurs = pd.concat(
    [
        indicateurs,
        pd.DataFrame(series_parts, index=indicateurs.index),
    ],
    axis=1,
)

In [15]:
#Controler les parts

def verifier_parts(table, colonnes, colonne_total):
    avec_observations = table[colonne_total].gt(0)

    valeurs = table.loc[avec_observations, colonnes]

    assert valeurs.notna().all().all()
    assert ((valeurs >= 0) & (valeurs <= 100)).all().all()

    assert np.allclose(
        valeurs.sum(axis=1).to_numpy(dtype=float),
        100.0,
    )

    assert table.loc[
        ~avec_observations, colonnes
    ].isna().all().all()


verifier_parts(
    indicateurs,
    list(PARTS_GROUPES.values()),
    COLONNE_TOTAL,
)

for groupe, codes in GROUPES.items():
    verifier_parts(
        indicateurs,
        [PARTS_INTRAGROUPE[code] for code in codes],
        COLONNES_GROUPES[groupe],
    )

print("Parts des groupes et parts intragroupe contrôlées.")

Parts des groupes et parts intragroupe contrôlées.


In [16]:
# Reprendre la population de référence

COLONNE_POPULATION = None
metadata_j10f = {}

if FICHIER_METADATA_J10F.is_file():
    with open(
        FICHIER_METADATA_J10F,
        "r",
        encoding="utf-8",
    ) as flux:
        metadata_j10f = json.load(flux)

if COLONNE_POPULATION is None:
    COLONNE_POPULATION = metadata_j10f.get("colonne_population")

if COLONNE_POPULATION is None:
    candidats = [
        colonne
        for colonne in [
            "POPULATION",
            "POPULATION_MUNICIPALE",
            "P23_POP",
            "P22_POP",
            "POP",
        ]
        if colonne in profil.columns
    ]

    if len(candidats) == 1:
        COLONNE_POPULATION = candidats[0]
    else:
        print("Candidats :", candidats)
        raise ValueError(
            "Renseigne COLONNE_POPULATION comme au J10f."
        )

if COLONNE_POPULATION not in profil.columns:
    raise ValueError("Colonne de population introuvable.")

population_par_code = (
    profil.set_index("CODGEO")[COLONNE_POPULATION]
    .pipe(convertir_nombre)
)

population = indicateurs["CODGEO"].map(population_par_code)

population_valide = population.gt(0).fillna(False)
denominateur = population.where(population_valide)

print("Population utilisée :", COLONNE_POPULATION)

Population utilisée : POPULATION_2022


In [17]:
#Calculer les 2 densités

DENSITES = {
    COLONNES_GROUPES[groupe]:
        f"DENSITE_OBS_{groupe}_10000_HAB"
    for groupe in GROUPES
}

for colonne_nombre, colonne_densite in DENSITES.items():
    indicateurs[colonne_densite] = (
        indicateurs[colonne_nombre]
        .div(denominateur)
        .mul(10_000)
    )

print(list(DENSITES.values()))

['DENSITE_OBS_HEBERGEMENTS_TOURISTIQUES_10000_HAB', 'DENSITE_OBS_SERVICES_TOURISTIQUES_10000_HAB']


In [18]:
#Agréger les capacités sans mélanger les unités

COLONNES_CAPACITES = {
    "G102": "NB_CHAMBRES_HOTELS_BPE2025",
    "G103": "NB_EMPLACEMENTS_PASSAGE_CAMPINGS_BPE2025",
    "G105": "NB_LOGEMENTS_AUTRES_HEBERGEMENTS_BPE2025",
}

COLONNES_STATUTS_CAPACITES = {
    "G102": "STATUT_CAPACITE_HOTELS",
    "G103": "STATUT_CAPACITE_CAMPINGS",
    "G105": "STATUT_CAPACITE_AUTRES_HEBERGEMENTS",
}

diagnostics_capacites = []

for code, colonne_capacite in COLONNES_CAPACITES.items():
    selection = tourisme.loc[
        tourisme["TYPEQU_ANALYSE"].eq(code)
    ]

    resume = (
        selection.groupby("CODGEO")
        .agg(
            NB_CAPACITES_CONNNUES=("CAPACITE_NUM", "count"),
            SOMME_CAPACITES_CONNNUES=(
                "CAPACITE_NUM",
                lambda serie: serie.sum(min_count=1),
            ),
        )
        .reset_index()
    )

    diagnostic = indicateurs[
        ["CODGEO", COLONNES_TYPES[code]]
    ].rename(
        columns={COLONNES_TYPES[code]: "NB_OBSERVATIONS"}
    ).merge(
        resume,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )

    diagnostic["NB_CAPACITES_CONNNUES"] = (
        diagnostic["NB_CAPACITES_CONNNUES"]
        .fillna(0)
        .astype("int64")
    )

    aucune_observation = diagnostic["NB_OBSERVATIONS"].eq(0)

    complet = (
        diagnostic["NB_CAPACITES_CONNNUES"]
        .eq(diagnostic["NB_OBSERVATIONS"])
    )

    diagnostic.loc[
        aucune_observation,
        "SOMME_CAPACITES_CONNNUES",
    ] = 0

    diagnostic["CAPACITE_TOTALE"] = (
        diagnostic["SOMME_CAPACITES_CONNNUES"].where(complet)
    )

    diagnostic["STATUT"] = "INCOMPLETE"

    diagnostic.loc[
        complet & ~aucune_observation,
        "STATUT",
    ] = "COMPLETE_DANS_EXTRACTION"

    diagnostic.loc[
        aucune_observation,
        "STATUT",
    ] = "AUCUNE_OBSERVATION_DANS_EXTRACTION"

    diagnostic["TYPEQU"] = code
    diagnostic["UNITE"] = UNITES_CAPACITE[code]

    diagnostic["COUVERTURE_CAPACITES_PCT"] = (
        diagnostic["NB_CAPACITES_CONNNUES"]
        .div(diagnostic["NB_OBSERVATIONS"].replace(0, np.nan))
        .mul(100)
    )

    ajout = diagnostic[
        ["CODGEO", "CAPACITE_TOTALE", "STATUT"]
    ].rename(columns={
        "CAPACITE_TOTALE": colonne_capacite,
        "STATUT": COLONNES_STATUTS_CAPACITES[code],
    })

    indicateurs = indicateurs.merge(
        ajout,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )

    diagnostics_capacites.append(diagnostic)

controle_capacites = pd.concat(
    diagnostics_capacites,
    ignore_index=True,
)

sauvegarder_csv(
    controle_capacites,
    DOSSIER_INTERIM / "controle_capacites_hebergements.csv",
)

display(controle_capacites.head(10))

,CODGEO,NB_OBSERVATIONS,NB_CAPACITES_CONNNUES,SOMME_CAPACITES_CONNNUES,CAPACITE_TOTALE,STATUT,TYPEQU,UNITE,COUVERTURE_CAPACITES_PCT
0,75056,1682,1682,88278,88278,COMPLETE_DANS_EXTRACTION,G102,CHAMBRES,100.0
1,77001,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
2,77002,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
3,77003,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
4,77004,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
5,77005,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
6,77006,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
7,77007,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
8,77008,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN
9,77009,0,0,0,0,AUCUNE_OBSERVATION_DANS_EXTRACTION,G102,CHAMBRES,NaN


In [19]:
# Joindre au profil 10G

nouvelles_colonnes = set(indicateurs.columns) - {"CODGEO"}
collisions = nouvelles_colonnes & set(profil.columns)

if collisions:
    raise ValueError(
        "Colonnes déjà présentes dans le profil : "
        + ", ".join(sorted(collisions))
    )

if indicateurs.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans les indicateurs.")

profil_j10g = profil.merge(
    indicateurs,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert len(profil_j10g) == len(profil)
assert profil_j10g["CODGEO"].is_unique

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10g[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

print("Colonnes avant :", len(profil.columns))
print("Colonnes après :", len(profil_j10g.columns))

Colonnes avant : 945
Colonnes après : 968


In [20]:
#Examiner les résultats

colonne_nom = next(
    (
        colonne
        for colonne in ["LIBELLE", "NOM_COMMUNE", "LIBGEO", "NOM_COM"]
        if colonne in profil_j10g.columns
    ),
    None,
)

colonnes_affichage = ["CODGEO"]

if colonne_nom:
    colonnes_affichage.append(colonne_nom)

colonnes_affichage += [
    COLONNE_POPULATION,
    "NB_OBS_HOTELS_BPE2025",
    "NB_CHAMBRES_HOTELS_BPE2025",
    "STATUT_CAPACITE_HOTELS",
    "NB_OBS_CAMPINGS_BPE2025",
    "NB_OBS_AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES_BPE2025",
    "NB_OBS_SERVICES_TOURISTIQUES_BPE2025",
]

print("Communes avec le plus d'observations d'hôtels :")

display(
    profil_j10g[colonnes_affichage]
    .sort_values("NB_OBS_HOTELS_BPE2025", ascending=False)
    .head(20)
)

print("Cergy et Pontoise :")

display(
    profil_j10g.loc[
        profil_j10g["CODGEO"].isin(["95127", "95500"]),
        colonnes_affichage,
    ]
)

bilan_categories = (
    tourisme.groupby(
        ["TYPEQU_ANALYSE", "CATEGORIE_ANALYSE", "GROUPE_ANALYSE"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    bilan_categories,
    DOSSIER_INTERIM / "bilan_categories.csv",
)

display(bilan_categories)

Communes avec le plus d'observations d'hôtels :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_HOTELS_BPE2025,NB_CHAMBRES_HOTELS_BPE2025,STATUT_CAPACITE_HOTELS,NB_OBS_CAMPINGS_BPE2025,NB_OBS_AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES_BPE2025,NB_OBS_SERVICES_TOURISTIQUES_BPE2025
0,75056,Paris,2113705.0,1682,88278,COMPLETE_DANS_EXTRACTION,1,100,2991
1225,95527,Roissy-en-France,2682.0,26,5867,COMPLETE_DANS_EXTRACTION,0,7,4
1027,93066,Saint-Denis,148907.0,22,2548,COMPLETE_DANS_EXTRACTION,0,1,56
965,92012,Boulogne-Billancourt,120205.0,20,1212,COMPLETE_DANS_EXTRACTION,0,5,89
979,92044,Levallois-Perret,68412.0,17,983,COMPLETE_DANS_EXTRACTION,0,3,64
971,92024,Clichy,65102.0,16,1290,COMPLETE_DANS_EXTRACTION,0,3,48
978,92040,Issy-les-Moulineaux,67695.0,16,1339,COMPLETE_DANS_EXTRACTION,0,3,38
754,78646,Versailles,83918.0,15,785,COMPLETE_DANS_EXTRACTION,1,0,49
984,92050,Nanterre,98119.0,15,1400,COMPLETE_DANS_EXTRACTION,0,3,29
973,92026,Courbevoie,81945.0,13,2465,COMPLETE_DANS_EXTRACTION,0,12,50


Cergy et Pontoise :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_HOTELS_BPE2025,NB_CHAMBRES_HOTELS_BPE2025,STATUT_CAPACITE_HOTELS,NB_OBS_CAMPINGS_BPE2025,NB_OBS_AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES_BPE2025,NB_OBS_SERVICES_TOURISTIQUES_BPE2025
1119,95127,Cergy,69578.0,10,854,COMPLETE_DANS_EXTRACTION,0,1,26
1220,95500,Pontoise,31623.0,1,91,COMPLETE_DANS_EXTRACTION,0,0,20


,TYPEQU_ANALYSE,CATEGORIE_ANALYSE,GROUPE_ANALYSE,NB_OBSERVATIONS
0,G101,AGENCES_VOYAGES,SERVICES_TOURISTIQUES,2498
1,G102,HOTELS,HEBERGEMENTS_TOURISTIQUES,2699
2,G103,CAMPINGS,HEBERGEMENTS_TOURISTIQUES,80
3,G104,INFORMATION_TOURISTIQUE,SERVICES_TOURISTIQUES,3571
4,G105,AUTRES_HEBERGEMENTS_COLLECTIFS_TOURISTIQUES,HEBERGEMENTS_TOURISTIQUES,272


In [21]:
#Créer le dictionnaire des indicateurs

lignes_dictionnaire = []

def documenter(colonne, description, unite, denominateur=""):
    lignes_dictionnaire.append({
        "COLONNE": colonne,
        "DESCRIPTION": description,
        "UNITE": unite,
        "DENOMINATEUR": denominateur,
    })


for code, libelle in TYPES.items():
    documenter(
        COLONNES_TYPES[code],
        f"Observations BPE du type {code} : {libelle}",
        "observations",
    )

    documenter(
        PARTS_INTRAGROUPE[code],
        f"Part du type {code} dans son groupe d'analyse",
        "%",
        COLONNES_GROUPES[TYPE_VERS_GROUPE[code]],
    )


for groupe in GROUPES:
    colonne_nombre = COLONNES_GROUPES[groupe]

    documenter(
        colonne_nombre,
        f"Observations du groupe analytique {groupe}",
        "observations",
    )

    documenter(
        PARTS_GROUPES[groupe],
        f"Part du groupe {groupe} dans J10g",
        "%",
        COLONNE_TOTAL,
    )

    documenter(
        DENSITES[colonne_nombre],
        f"Densité du groupe {groupe}",
        "observations pour 10 000 habitants",
        COLONNE_POPULATION,
    )


documenter(
    COLONNE_TOTAL,
    "Total des observations touristiques retenues",
    "observations",
)

for code, colonne in COLONNES_CAPACITES.items():
    documenter(
        colonne,
        (
            f"Somme des capacités du type {code}, "
            "renseignée uniquement si toutes les capacités sont connues"
        ),
        UNITES_CAPACITE[code],
    )

    documenter(
        COLONNES_STATUTS_CAPACITES[code],
        f"Complétude des capacités du type {code} dans l'extraction",
        "statut",
    )

dictionnaire = pd.DataFrame(lignes_dictionnaire)

assert dictionnaire["COLONNE"].is_unique
assert set(dictionnaire["COLONNE"]) == (
    set(indicateurs.columns) - {"CODGEO"}
)

sauvegarder_csv(
    dictionnaire,
    DOSSIER_INTERIM / "dictionnaire_indicateurs_j10g.csv",
)

In [22]:
#Contrôles finaux

colonnes_effectifs = [
    *COLONNES_TYPES.values(),
    *COLONNES_GROUPES.values(),
    COLONNE_TOTAL,
]

assert len(profil_j10g) == len(profil)
assert profil_j10g["CODGEO"].is_unique
assert set(profil_j10g["CODGEO"]) == codes_profil

assert tourisme["ID_OBSERVATION"].is_unique
assert tourisme["CODGEO"].isin(codes_profil).all()

assert indicateurs[colonnes_effectifs].notna().all().all()
assert indicateurs[colonnes_effectifs].ge(0).all().all()

assert indicateurs[COLONNE_TOTAL].sum() == len(tourisme)

assert (
    indicateurs[colonnes_types].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

for colonne in DENSITES.values():
    valeurs = indicateurs[colonne].dropna().astype(float)

    assert np.isfinite(valeurs).all()
    assert valeurs.ge(0).all()

    assert indicateurs.loc[
        ~population_valide, colonne
    ].isna().all()

assert (
    controle_capacites["NB_CAPACITES_CONNNUES"]
    <= controle_capacites["NB_OBSERVATIONS"]
).all()

for code, colonne in COLONNES_CAPACITES.items():
    colonne_statut = COLONNES_STATUTS_CAPACITES[code]

    incomplet = indicateurs[colonne_statut].eq("INCOMPLETE")

    assert indicateurs.loc[incomplet, colonne].isna().all()
    assert indicateurs.loc[~incomplet, colonne].notna().all()

    valeurs = indicateurs[colonne].dropna().astype(float)
    assert np.isfinite(valeurs).all()
    assert valeurs.ge(0).all()

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10g[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

controle = pd.DataFrame([{
    "COMMUNES_ENTREE": len(profil),
    "COMMUNES_SORTIE": len(profil_j10g),
    "OBSERVATIONS_TOURISME": len(tourisme),
    "OBSERVATIONS_HORS_PROFIL": len(hors_profil),
    "OBSERVATIONS_SIRET_TYPE_REPETE": len(repetitions),
    "COUPLES_COMMUNE_TYPE_CAPACITE_INCOMPLETE": int(
        controle_capacites["STATUT"].eq("INCOMPLETE").sum()
    ),
    "ETAT": (
        "CONTROLES_STRUCTURELS_OK"
        if repetitions.empty
        else "CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER"
    ),
}])

sauvegarder_csv(
    controle,
    DOSSIER_INTERIM / "controle_jointures.csv",
)

display(controle)

,COMMUNES_ENTREE,COMMUNES_SORTIE,OBSERVATIONS_TOURISME,OBSERVATIONS_HORS_PROFIL,OBSERVATIONS_SIRET_TYPE_REPETE,COUPLES_COMMUNE_TYPE_CAPACITE_INCOMPLETE,ETAT
0,1266,1266,9120,0,23,1,CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER


In [23]:
#Enregistrer les résultats

sauvegarder_csv(tourisme, FICHIER_DETAIL)
sauvegarder_csv(indicateurs, FICHIER_COMMUNES)
sauvegarder_csv(profil_j10g, FICHIER_SORTIE)

metadonnees = {
    "notebook": "10g_tourisme_hebergements_idf.ipynb",
    "date_traitement_utc": DATE_TRAITEMENT,
    "source": "INSEE_BPE_2025",
    "url_nomenclature": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_liste_hierarchisee_TYPEQU.html"
    ),
    "url_dictionnaire": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_anonymisee_dictionnaire_variables.html"
    ),
    "fichier_bpe": str(FICHIER_BPE),
    "sha256_bpe": HASH_BPE,
    "profil_entree": str(FICHIER_PROFIL),
    "sha256_profil_entree": sha256_fichier(FICHIER_PROFIL),
    "profil_sortie": str(FICHIER_SORTIE),
    "colonne_population": COLONNE_POPULATION,
    "types": TYPES,
    "groupes_analytiques": GROUPES,
    "origine_groupes": "REGROUPEMENT_PROJET_NON_OFFICIEL",
    "unites_capacite": UNITES_CAPACITE,
    "nombre_observations": len(tourisme),
    "nombre_communes": len(profil_j10g),
    "observations_siret_type_repete": len(repetitions),
    "deduplication_automatique": False,
    "parts": {
        "groupes": "Dans le total J10g",
        "sous_categories": "Dans leur groupe uniquement",
    },
    "densites": "Groupes uniquement, pour 10 000 habitants",
    "limites": [
        "Comptages d'observations BPE.",
        "Ouverture actuelle non vérifiée.",
        "Périmètre limité des catégories touristiques BPE.",
        "Aucune donnée de nuitées ou de taux d'occupation.",
        "Chambres, emplacements et logements non additionnés.",
        "Totaux de capacité manquants en cas de couverture incomplète.",
        "Complétude calculée dans l'extraction, pas preuve d'exhaustivité.",
        "Coordonnées contrôlées par rectangle uniquement.",
    ],
}

with open(
    DOSSIER_INTERIM / "metadata_j10g.json",
    "w",
    encoding="utf-8",
) as flux:
    json.dump(
        metadonnees,
        flux,
        ensure_ascii=False,
        indent=2,
    )

print("Détail :", FICHIER_DETAIL)
print("Indicateurs communaux :", FICHIER_COMMUNES)
print("Profil enrichi :", FICHIER_SORTIE)
print("J10g enregistré.")

Détail : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\tourisme_hebergements_idf_j10g.csv
Indicateurs communaux : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10g\tourisme_hebergements_communes.csv
Profil enrichi : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10g.csv
J10g enregistré.
